# Use case: triage a support inbox

A small shop has help articles in three areas: **billing**, **shipping** and **account**. Tickets
arrive as free text. Three things need to happen to each one:

1. **Route** it to the right area (a zero-shot classifier: no training data, just the articles).
2. **Draft** a reply from the matching article, with a citation.
3. **Escalate** to a human when no article is close enough, instead of guessing.

A `library()` with one topic per area does all three: `route()` is the classifier, `ask()` finds
the article, `answer(refuse_below=...)` is the escalation rule. Twelve labelled tickets measure it.

**Needs:** Ollama with `nomic-embed-text`. The reply-drafting step also needs `qwen2.5:7b-instruct`.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STORE = ROOT / ".usecase_nb" / "support"          # the store lives here; delete the folder to start over
RUN_LLM = True                               # the steps that call a chat model are slow on CPU
LLM = "qwen2.5:7b-instruct"                      # follows context better than llama3.2:3b

from slim_llm_memory import library

db = library(STORE)

db.topic("billing").add({
    "refunds.md":  "Refunds are issued to the original payment method within 5 to 7 business days of approval. "
                   "Partial refunds are possible for damaged items; contact support with a photo.",
    "invoices.md": "Invoices are emailed after every order and can be downloaded from Orders > Invoice as a PDF. "
                   "Company name and VAT number can be edited before the order ships.",
    "cards.md":    "We accept Visa, Mastercard and PayPal. A declined card is usually a bank-side block on "
                   "online purchases; try again after calling your bank or use PayPal.",
})
db.topic("shipping").add({
    "delivery.md": "Standard delivery takes 3 to 5 business days, express 1 to 2. Orders placed before 14:00 "
                   "ship the same day. A tracking link is emailed as soon as the parcel leaves the warehouse.",
    "lost.md":     "If tracking has not moved for 7 days the parcel counts as lost. We send a replacement "
                   "at no cost or refund the order, your choice.",
    "returns.md":  "Returns are free within 30 days. Print the label from Orders > Return, put it on the box "
                   "and drop it at any parcel shop.",
})
db.topic("account").add({
    "password.md": "Use Forgot password on the login page. The reset link is valid for one hour. "
                   "If the email does not arrive, check spam and confirm the address you registered with.",
    "delete.md":   "You can delete your account under Settings > Privacy. Order history is kept for 10 years "
                   "for tax reasons but is no longer linked to a person.",
    "twofactor.md": "Two-factor authentication uses an authenticator app. Lost your phone? Use one of the "
                    "backup codes shown when you enabled it, or contact support with a copy of your ID.",
})
db

library(/home/trbck/workspace/slim-llm-memory/.usecase_nb/support, ollama:nomic-embed-text): 3 active, 0 archived
  account      3 doc(s)       3 chunks  active
  billing      3 doc(s)       3 chunks  active
  shipping     3 doc(s)       3 chunks  active

## 1. Routing as classification

`route()` compares the ticket with each topic's centroid, the mean of its vectors. One embedding
call, one tiny matrix product, no model trained. Twelve tickets, hand-labelled, none of which
quote an article word for word.

In [2]:
TICKETS = [
    ("My card keeps getting rejected at checkout, what can I do?",           "billing"),
    ("I need the invoice to show my company's VAT id",                      "billing"),
    ("Still waiting for the money back on the order I cancelled",           "billing"),
    ("Can I pay with PayPal?",                                              "billing"),
    ("Where is my parcel? The tracking page hasn't changed in over a week", "shipping"),
    ("How do I send something back, it doesn't fit",                        "shipping"),
    ("If I order this morning will it arrive tomorrow?",                    "shipping"),
    ("The box arrived crushed and the item inside is broken",               "shipping"),
    ("I can't log in, the reset email never shows up",                      "account"),
    ("I got a new phone and my authenticator app is gone",                  "account"),
    ("Please remove all my data, I don't want an account anymore",          "account"),
    ("Can I change the email address on my profile?",                       "account"),
]

right = 0
print(f"{'ticket':<70} {'routed to':<10} {'expected':<10}")
for text, expected in TICKETS:
    rt = db.route(text, m=1)
    got = rt.chosen[0] if rt.chosen else "—"
    right += got == expected
    print(f"{text[:68]:<70} {got:<10} {expected:<10} {'' if got == expected else '✗'}")
print(f"\nrouting accuracy: {right}/{len(TICKETS)}")

ticket                                                                 routed to  expected  


My card keeps getting rejected at checkout, what can I do?             billing    billing    


I need the invoice to show my company's VAT id                         billing    billing    


Still waiting for the money back on the order I cancelled              billing    billing    


Can I pay with PayPal?                                                 billing    billing    


...

The box arrived crushed and the item inside is broken                  billing    shipping   ✗


I can't log in, the reset email never shows up                         account    account    


I got a new phone and my authenticator app is gone                     account    account    


Please remove all my data, I don't want an account anymore             account    account    


Can I change the email address on my profile?                          account    account    

routing accuracy: 10/12


A wrong route is not a wrong answer yet. `ask()` searches every topic anyway when the library is
small, and each hit names its topic, so the article itself is the real classifier. Compare the
topic of the **best chunk** with the label:

In [3]:
right = 0
for text, expected in TICKETS:
    r = db.ask(text, k=1, min_score=0.0)
    got = r.top.meta["topic"] if r.top else "—"
    right += got == expected
print(f"best-chunk accuracy: {right}/{len(TICKETS)}")

best-chunk accuracy: 11/12


## 2. Measure before choosing the escalation threshold

`refuse_below` is a cosine score, so look at real scores before picking it. Four ordinary
tickets, one about something the shop has no article for, and one that is not about the shop
at all:

In [4]:
for text in [t for t, _ in TICKETS[:4]] + ["Do you offer gift wrapping for birthdays?",
                                             "What is the capital of France?"]:
    r = db.ask(text, k=1, min_score=0.0)
    print(f"{r.top.score:.2f}  {text}")

0.72  My card keeps getting rejected at checkout, what can I do?


0.74  I need the invoice to show my company's VAT id


0.59  Still waiting for the money back on the order I cancelled


0.61  Can I pay with PayPal?


0.53  Do you offer gift wrapping for birthdays?


0.35  What is the capital of France?


The gift-wrapping ticket lands just under the weakest real ticket. That gap is thin with nine
articles; it widens as the knowledge base grows, because real tickets find closer matches. The
threshold goes into the gap: `REFUSE_BELOW = 0.57`.

## 3. Draft a reply, or escalate

`answer(refuse_below=...)` skips the model and returns a refusal when the best hit's cosine is
under the threshold. Two ordinary tickets get drafted replies with the article cited; the
gift-wrapping ticket is escalated without an LLM call, and without an invented policy.

In [5]:
REFUSE_BELOW = 0.57

def handle(ticket):
    a = db.answer(ticket, model=LLM, k=3, refuse_below=REFUSE_BELOW)
    if a.refused:
        best = f"{a.hits[0].meta['topic']}/{a.hits[0].meta['doc']} at {a.hits[0].score:.2f}" if a.hits else "nothing"
        return f"→ ESCALATE to a human (best match: {best})"
    src = ", ".join(f"{a.hits[i - 1].meta['topic']}/{a.hits[i - 1].meta['doc']}" for i in a.citations)
    return f"→ DRAFT: {a}\n   sources: {src}"

if RUN_LLM:
    for ticket in ["Where is my parcel? The tracking page hasn't changed in over a week",
                   "I got a new phone and my authenticator app is gone",
                   "Do you offer gift wrapping for birthdays?"]:
        print(ticket)
        print(handle(ticket), "\n")

Where is my parcel? The tracking page hasn't changed in over a week


→ DRAFT: Based on [1], if your tracking page hasn't changed in over a week, your parcel counts as lost and you can request a replacement or refund.
   sources: shipping/lost.md 

I got a new phone and my authenticator app is gone


→ DRAFT: Use one of the backup codes shown when you enabled two-factor authentication, or contact support with a copy of your ID [1].
   sources: account/twofactor.md 

Do you offer gift wrapping for birthdays?


→ ESCALATE to a human (best match: shipping/returns.md at 0.53) 



## Takeaways

- **Articles are the training data.** Adding a help article is `db.topic(area).add(...)`; the classifier updates itself.
- **Route for the label, ask for the answer.** Centroids are coarse; the best chunk is the precise signal.
- **Refuse below a score you measured.** Print the scores of a few good and bad tickets, then pick the gap. Without it, the model answers the gift-wrapping ticket with "the context does not say" and three irrelevant citations.
- **Next step:** log every escalated ticket, write the missing article, add it. The next similar ticket is answered.

In [6]:
db.close()